# Phase 3 Analysis — MIT-BIH Real ECG Dataset

This notebook extends the Phase 2 synthetic analysis with:
- **R-peak integrity** scoring per record and configuration
- **Per-record ΔSNR** breakdown across the MIT-BIH test set
- **Comparison** of synthetic vs. real-data performance

## Prerequisites

Run the Phase 3 experiment first:
```bash
uv run neuroflux-experiment run configs/phase3/quick_validation.yaml --output-db results/phase3_quick.db
```

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

DB_PATH = Path("../results/phase3_quick.db")
assert DB_PATH.exists(), f"Run the experiment first: {DB_PATH}"

## 1. Load Results

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

rows = conn.execute("SELECT * FROM runs WHERE dataset = 'mit-bih'").fetchall()

print(f"Total MIT-BIH runs: {len(rows)}")
if rows:
    print(f"Columns: {rows[0].keys()}")

## 2. Per-Record ΔSNR Breakdown

In [ ]:
from collections import defaultdict

record_snrs: dict[str, list[float]] = defaultdict(list)
for row in rows:
    record_snrs[row["record_id"]].append(row["delta_snr"])

records = sorted(record_snrs.keys())
means = [np.mean(record_snrs[r]) for r in records]
stds = [np.std(record_snrs[r]) for r in records]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(records)), means, yerr=stds, capsize=4, alpha=0.8)
ax.set_xticks(range(len(records)))
ax.set_xticklabels(records, rotation=45)
ax.set_xlabel("MIT-BIH Record ID")
ax.set_ylabel("ΔSNR (dB)")
ax.set_title("Per-Record ΔSNR on MIT-BIH")
ax.axhline(y=0, color="red", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 3. R-Peak Integrity Analysis

In [ ]:
record_rpeak: dict[str, list[float]] = defaultdict(list)
for row in rows:
    if row["rpeak_integrity"] is not None:
        record_rpeak[row["record_id"]].append(row["rpeak_integrity"])

if record_rpeak:
    records_rp = sorted(record_rpeak.keys())
    means_rp = [np.mean(record_rpeak[r]) for r in records_rp]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(records_rp)), means_rp, alpha=0.8, color="teal")
    ax.set_xticks(range(len(records_rp)))
    ax.set_xticklabels(records_rp, rotation=45)
    ax.set_xlabel("MIT-BIH Record ID")
    ax.set_ylabel("R-Peak Integrity")
    ax.set_title("R-Peak Integrity per Record")
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.9, color="green", linestyle="--", alpha=0.5, label="90% target")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No R-peak integrity data available.")

## 4. ΔSNR vs R-Peak Integrity Scatter

In [ ]:
delta_snrs = [row["delta_snr"] for row in rows if row["rpeak_integrity"] is not None]
rpeaks = [row["rpeak_integrity"] for row in rows if row["rpeak_integrity"] is not None]

if delta_snrs:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(delta_snrs, rpeaks, alpha=0.6, edgecolors="black", linewidths=0.5)
    ax.set_xlabel("ΔSNR (dB)")
    ax.set_ylabel("R-Peak Integrity")
    ax.set_title("ΔSNR vs R-Peak Integrity")
    ax.axhline(y=0.9, color="green", linestyle="--", alpha=0.5)
    ax.axvline(x=3.0, color="blue", linestyle="--", alpha=0.5, label="3 dB target")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No data for scatter plot.")

## 5. Summary Statistics

In [ ]:
all_dsnr = [row["delta_snr"] for row in rows]
all_rpeak = [
    row["rpeak_integrity"] for row in rows if row["rpeak_integrity"] is not None
]

print("=== Phase 3 Summary ===")
print(f"Total runs:           {len(rows)}")
print(f"Unique records:       {len(set(row['record_id'] for row in rows))}")
print(f"Mean ΔSNR:            {np.mean(all_dsnr):.3f} ± {np.std(all_dsnr):.3f} dB")
print(f"ΔSNR ≥ 3 dB:          {sum(1 for d in all_dsnr if d >= 3.0)} / {len(all_dsnr)}")
if all_rpeak:
    rpeak_mean, rpeak_std = np.mean(all_rpeak), np.std(all_rpeak)
    n_rpeak_ok = sum(1 for r in all_rpeak if r >= 0.9)
    print(f"Mean R-peak integrity: {rpeak_mean:.3f} ± {rpeak_std:.3f}")
    print(f"R-peak ≥ 0.9:         {n_rpeak_ok} / {len(all_rpeak)}")

conn.close()